# FitTrack: Label Reader -- OCR Cleanup + Unhealthy-Ingredient & Calorie Analysis

End-to-end pipeline for the work described in
[`docs/spec-label-reader.md`](../docs/spec-label-reader.md).

This notebook only trains **Stage 2** (the structuring/health-reasoning
LLM). Per the spec, **Stage 1 (OCR)** uses an existing pretrained model
(`zai-org/GLM-OCR`) as-is, with no training -- it only shows up here in an
optional end-to-end demo cell near the end, not in the training loop.

Pipeline:

1. Stream a filtered sample of products from the Open Food Facts bulk export
2. Build rule-based "unhealthy ingredient" labels from OFF's own fields
3. Synthesize OCR-noise on the clean ingredient text (see the note in
   Section 4 -- this is a deliberate simplification, flagged for follow-up)
4. Build an instruction-tuning dataset (raw/noisy text -> structured JSON)
5. Load `Qwen2.5-3B-Instruct` in 4-bit (QLoRA)
6. Apply LoRA
7. Train
8. Save the adapter
9. Evaluate (ingredient recall, flag accuracy, calorie MAE/MAPE)
10. Optional: chain with GLM-OCR for a real photo -> structured-output demo

**Run this on Colab with a GPU runtime.** Free T4 should be enough for a 3B
model in 4-bit.

As with the calorie-estimator notebook, a few cells have `# VERIFY:` /
`# NOTE:` comments flagging assumptions that weren't re-checked against a
live data pull while writing this -- read those before trusting the
numbers.

## 1. Setup

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl \
    datasets pandas requests rapidfuzz huggingface_hub

In [ ]:
import os
import json
import gzip
import random
import re
from pathlib import Path
from getpass import getpass

import requests
import torch
import pandas as pd

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU detected -- training will be impractically slow. "
          "Switch the Colab runtime to GPU.")

from huggingface_hub import login
hf_token = getpass("Hugging Face token (leave blank to skip login): ")
if hf_token:
    login(hf_token)

## 2. Stream a filtered sample from Open Food Facts

The full bulk export (`openfoodfacts-products.jsonl.gz`) is multiple
gigabytes -- too large to fully download for this pipeline. Instead, stream
the gzip response line-by-line and stop once we have enough products that
pass the filter (non-empty `ingredients_text`, a usable
`energy-kcal_100g`/`_serving` value, English language, reasonable text
length).

`# VERIFY:` the export URL and field names below against
https://world.openfoodfacts.org/data before relying on them long-term -- OFF
has changed its export format/location before.

This streams over the network and stops early, but still has to read
through the file from the start, so it can take a few minutes even though
we only keep a small sample.

In [ ]:
OFF_EXPORT_URL = "https://static.openfoodfacts.org/data/openfoodfacts-products.jsonl.gz"
TARGET_SAMPLE_SIZE = 15000  # products to keep after filtering

def usable_calories(nutriments):
    for key in ("energy-kcal_100g", "energy-kcal_serving"):
        val = nutriments.get(key)
        if isinstance(val, (int, float)) and val > 0:
            return float(val)
    return None

def passes_filter(product):
    text = (product.get("ingredients_text") or "").strip()
    if len(text) < 20 or len(text) > 2000:
        return False
    if product.get("lang") not in (None, "en") and product.get("ingredients_text_en") is None:
        # Prefer English-language ingredient text where available.
        if not text.isascii():
            return False
    return usable_calories(product.get("nutriments", {})) is not None

def stream_off_sample(url, target_size):
    kept = []
    with requests.get(url, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        with gzip.GzipFile(fileobj=resp.raw) as gz:
            for i, line in enumerate(gz):
                if len(kept) >= target_size:
                    break
                try:
                    product = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if passes_filter(product):
                    kept.append({
                        "code": product.get("code"),
                        "product_name": product.get("product_name") or "",
                        "ingredients_text": product["ingredients_text"].strip(),
                        "calories_per_serving": usable_calories(product["nutriments"]),
                        "additives_tags": product.get("additives_tags", []),
                        "nutriscore_grade": product.get("nutriscore_grade"),
                        "nova_group": product.get("nova_group"),
                    })
                if i % 200000 == 0:
                    print(f"...scanned {i} lines, kept {len(kept)}")
    return kept

Path("./data/off").mkdir(parents=True, exist_ok=True)
raw_path = Path("./data/off/sample_raw.jsonl")

if raw_path.exists():
    print(f"{raw_path} already exists, skipping re-download")
else:
    sample = stream_off_sample(OFF_EXPORT_URL, TARGET_SAMPLE_SIZE)
    with open(raw_path, "w") as f:
        for row in sample:
            f.write(json.dumps(row) + "\n")
    print(f"Kept {len(sample)} products -> {raw_path}")

products_df = pd.read_json(raw_path, lines=True)
print(f"Loaded {len(products_df)} products from disk")
products_df.head()

## 3. Rule-based "unhealthy ingredient" flagging

Generates health-flag labels programmatically from OFF's own fields and a
curated keyword list, rather than hand-labeling -- per the spec.
`# VERIFY:` this keyword list is a starting point, not a complete or
medically-reviewed list; treat flags as informational.

In [ ]:
UNHEALTHY_KEYWORDS = {
    "high fructose corn syrup": "added sugar linked to metabolic issues",
    "corn syrup": "added sugar",
    "partially hydrogenated": "trans fat, linked to cardiovascular risk",
    "hydrogenated oil": "may contain trans fat",
    "trans fat": "linked to cardiovascular risk",
    "aspartame": "artificial sweetener, debated long-term effects",
    "sucralose": "artificial sweetener",
    "acesulfame": "artificial sweetener",
    "monosodium glutamate": "flavor enhancer, sensitivity in some people",
    "sodium nitrite": "preservative, linked to processed-meat health concerns",
    "sodium benzoate": "preservative",
    "potassium bromate": "flour treatment agent, health concerns",
    "bha": "preservative, potential carcinogen at high doses",
    "bht": "preservative",
    "red 40": "artificial dye, linked to hyperactivity in some studies",
    "yellow 5": "artificial dye",
    "yellow 6": "artificial dye",
}

def flag_ingredients(ingredients_text, additives_tags, nutriscore_grade, nova_group):
    text_lower = ingredients_text.lower()
    flags = []
    for keyword, reason in UNHEALTHY_KEYWORDS.items():
        if keyword in text_lower:
            flags.append({"ingredient": keyword, "reason": reason})

    if nova_group == 4:
        flags.append({"ingredient": "(overall product)", "reason": "NOVA group 4: ultra-processed food"})
    if nutriscore_grade in ("d", "e"):
        flags.append({"ingredient": "(overall product)", "reason": f"Nutri-Score {nutriscore_grade.upper()}: lower nutritional quality"})

    return flags

products_df["flags"] = products_df.apply(
    lambda r: flag_ingredients(r["ingredients_text"], r["additives_tags"], r["nutriscore_grade"], r["nova_group"]),
    axis=1,
)
products_df["num_flags"] = products_df["flags"].apply(len)
print(products_df["num_flags"].value_counts().sort_index())

## 4. Synthetic OCR noise on the clean ingredient text

**NOTE:** Open Food Facts' bulk export doesn't expose a simple "raw OCR
text" column paired with the cleaned `ingredients_text` -- the raw OCR
result lives per-image on their image server, not in this export, and
crawling it at scale is a separate, heavier pipeline. As a practical
stand-in, this section injects realistic character-level OCR error
patterns into the *clean* text to produce a synthetic noisy input.

This is a deliberate simplification, not a claim that it matches real OCR
error distributions. If accuracy on real photographed labels turns out
poor after deployment, revisit this by actually running an OCR engine over
OFF's `image_ingredients_url` photos (where present) to get genuine
noisy/clean pairs, at the cost of a slower, image-downloading data-prep
stage.

In [ ]:
OCR_CONFUSIONS = [
    ("O", "0"), ("o", "0"), ("l", "1"), ("I", "1"), ("S", "5"),
    ("B", "8"), ("rn", "m"), ("cl", "d"), ("ii", "n"),
]

def inject_ocr_noise(text, p_char_drop=0.01, p_space_drop=0.03, p_case_flip=0.02, p_confusion=0.05):
    chars = list(text)
    out = []
    for ch in chars:
        if ch == " " and random.random() < p_space_drop:
            continue  # dropped space -- common OCR artifact on dense label text
        if ch.isalpha() and random.random() < p_case_flip:
            ch = ch.swapcase()
        if random.random() < p_char_drop:
            continue  # dropped character
        out.append(ch)
    noisy = "".join(out)

    for orig, sub in OCR_CONFUSIONS:
        if random.random() < p_confusion:
            noisy = noisy.replace(orig, sub)

    return noisy

random.seed(SEED)
products_df["ocr_noisy_text"] = products_df["ingredients_text"].apply(inject_ocr_noise)
products_df[["ingredients_text", "ocr_noisy_text"]].head(3)

## 5. Build the instruction-tuning dataset

Same `instruction` / `input` / `output` shape as the spec doc.

In [ ]:
INSTRUCTION = (
    "Given this OCR'd label text, list the ingredients, flag any unhealthy "
    "ones with a short reason, and state the calories per serving."
)

def to_example(row):
    output = {
        "ingredients": [i.strip() for i in row["ingredients_text"].split(",") if i.strip()],
        "flags": row["flags"],
        "calories_per_serving": row["calories_per_serving"],
    }
    return {
        "code": row["code"],
        "instruction": INSTRUCTION,
        "input": row["ocr_noisy_text"],
        "output": json.dumps(output),
    }

examples = products_df.apply(to_example, axis=1).tolist()
random.shuffle(examples)

n = len(examples)
train_examples = examples[: int(n * 0.8)]
val_examples = examples[int(n * 0.8): int(n * 0.9)]
test_examples = examples[int(n * 0.9):]

Path("./data/processed").mkdir(parents=True, exist_ok=True)
for name, subset in [("train", train_examples), ("val", val_examples), ("test", test_examples)]:
    path = f"./data/processed/{name}.jsonl"
    with open(path, "w") as f:
        for ex in subset:
            f.write(json.dumps(ex) + "\n")
    print(f"{name}: {len(subset)} examples -> {path}")

## 6. Sanity check a sample

In [ ]:
with open("./data/processed/train.jsonl") as f:
    sample = json.loads(f.readline())
print(json.dumps(sample, indent=2))

## 7. Load the base model in 4-bit (QLoRA)

`Qwen2.5-3B-Instruct` -- open weights with no gated-access request needed
(unlike Llama-3.2-3B-Instruct, which requires accepting Meta's license on
the Hub before download). Swap to Llama-3.2-3B-Instruct if you've already
got access and prefer it; the rest of this notebook doesn't depend on which
one you pick.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False

## 8. Apply LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 9. Dataset & tokenization

Builds the chat-formatted input via the tokenizer's chat template, masking
the prompt portion of `labels` with `-100` so loss is only computed on the
assistant's JSON response.

In [ ]:
from torch.utils.data import Dataset

MAX_LENGTH = 1024

class LabelReaderDataset(Dataset):
    def __init__(self, jsonl_path):
        with open(jsonl_path) as f:
            self.examples = [json.loads(line) for line in f]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        user_content = f"{ex['instruction']}\n\nLabel text:\n{ex['input']}"
        prompt_messages = [{"role": "user", "content": user_content}]
        full_messages = prompt_messages + [{"role": "assistant", "content": ex["output"]}]

        prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
        full_text = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)

        full_ids = tokenizer(full_text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")["input_ids"][0]
        prompt_len = len(tokenizer(prompt_text, add_special_tokens=False)["input_ids"])

        labels = full_ids.clone()
        labels[:prompt_len] = -100

        return {"input_ids": full_ids, "labels": labels}

def collate_fn(batch):
    max_len = max(len(b["input_ids"]) for b in batch)
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)

    for i, b in enumerate(batch):
        length = len(b["input_ids"])
        input_ids[i, :length] = b["input_ids"]
        labels[i, :length] = b["labels"]
        attention_mask[i, :length] = 1

    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

train_dataset = LabelReaderDataset("./data/processed/train.jsonl")
val_dataset = LabelReaderDataset("./data/processed/val.jsonl")
test_dataset = LabelReaderDataset("./data/processed/test.jsonl")
print(f"train: {len(train_dataset)}  val: {len(val_dataset)}  test: {len(test_dataset)}")

## 10. Training

Text-only and smaller per-example than the VLM track, so this trains
faster on the same GPU -- per the spec, well under a day even over the full
~12k-example train split.

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./checkpoints/qwen2.5-label-reader-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1.5e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
)

trainer.train()

## 11. Save the LoRA adapter

In [ ]:
ADAPTER_DIR = "./checkpoints/qwen2.5-label-reader-lora-final"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved adapter to {ADAPTER_DIR}")

# model.push_to_hub("YOUR_HF_USERNAME/qwen2.5-label-reader-lora")
# tokenizer.push_to_hub("YOUR_HF_USERNAME/qwen2.5-label-reader-lora")

## 12. Evaluate on the held-out test set

Three checks per the spec: ingredient-list overlap, flag accuracy against
the rule-based ground truth, and calorie-parsing MAE/MAPE.

In [ ]:
from rapidfuzz import fuzz

model.eval()

def generate(input_text):
    user_content = f"{INSTRUCTION}\n\nLabel text:\n{input_text}"
    messages = [{"role": "user", "content": user_content}]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=400, do_sample=False)

    generated = tokenizer.decode(output_ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated

def safe_parse(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                return None
        return None

rows = []
for ex in test_dataset.examples:
    raw_output = generate(ex["input"])
    parsed = safe_parse(raw_output)
    true_output = json.loads(ex["output"])

    ingredient_overlap = None
    flag_match = None
    pred_kcal = None

    if parsed:
        pred_ingredients = " ".join(parsed.get("ingredients", []))
        true_ingredients = " ".join(true_output.get("ingredients", []))
        ingredient_overlap = fuzz.token_set_ratio(pred_ingredients, true_ingredients)

        pred_flags = {f.get("ingredient", "") for f in parsed.get("flags", [])}
        true_flags = {f.get("ingredient", "") for f in true_output.get("flags", [])}
        flag_match = len(pred_flags & true_flags) / max(len(true_flags), 1)

        pred_kcal = parsed.get("calories_per_serving")

    rows.append({
        "code": ex["code"],
        "true_kcal": true_output.get("calories_per_serving"),
        "pred_kcal": pred_kcal,
        "ingredient_overlap_pct": ingredient_overlap,
        "flag_recall": flag_match,
        "raw_output": raw_output,
    })

eval_df = pd.DataFrame(rows)
valid_kcal = eval_df.dropna(subset=["true_kcal", "pred_kcal"])
mae = (valid_kcal["true_kcal"] - valid_kcal["pred_kcal"]).abs().mean()
mape = ((valid_kcal["true_kcal"] - valid_kcal["pred_kcal"]).abs() / valid_kcal["true_kcal"]).mean() * 100

print(f"Parseable outputs: {eval_df['pred_kcal'].notna().sum()}/{len(eval_df)}")
print(f"Calorie MAE:  {mae:.1f} kcal")
print(f"Calorie MAPE: {mape:.1f}%")
print(f"Mean ingredient overlap (fuzzy token set ratio): {eval_df['ingredient_overlap_pct'].mean():.1f}")
print(f"Mean flag recall: {eval_df['flag_recall'].mean():.2f}")

eval_df.to_csv("./data/processed/eval_results.csv", index=False)
eval_df.head(10)

## 13. Optional: end-to-end demo with GLM-OCR on a real photo

Chains Stage 1 (OCR) and Stage 2 (this fine-tuned model) on an actual label
photo, rather than the synthetic-noise text used for training.

`# VERIFY:` the exact `transformers` API for GLM-OCR -- this uses the
generic `AutoModelForImageTextToText` / `AutoProcessor` classes with
`trust_remote_code=True`, which should work but wasn't run against a live
checkpoint while writing this notebook. Check
https://huggingface.co/docs/transformers/main/en/model_doc/glm_ocr for the
current recommended loading code before relying on this cell.

Skip this section if you just want the Stage 2 training/eval results above;
GLM-OCR itself is not trained or modified here.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor as OCRProcessor
from PIL import Image

OCR_MODEL_ID = "zai-org/GLM-OCR"

ocr_processor = OCRProcessor.from_pretrained(OCR_MODEL_ID, trust_remote_code=True)
ocr_model = AutoModelForImageTextToText.from_pretrained(
    OCR_MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16, device_map="auto"
)

def run_ocr(image_path):
    image = Image.open(image_path)
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": "Read all the text in this image."},
    ]}]
    prompt_text = ocr_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = ocr_processor(text=[prompt_text], images=[image], return_tensors="pt").to(ocr_model.device)
    with torch.no_grad():
        output_ids = ocr_model.generate(**inputs, max_new_tokens=512)
    return ocr_processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )[0]

# Example usage once you have a real label photo to test with:
# raw_text = run_ocr("./my_label_photo.jpg")
# print(raw_text)
# print(generate(raw_text))

## 14. Next steps

- If ingredient overlap / flag recall is low, the synthetic-noise
  simplification (Section 4) is the most likely culprit -- consider
  building real noisy/clean pairs from OFF's `image_ingredients_url`
  photos run through an actual OCR engine, per the NOTE there.
- Hosting both GLM-OCR and this fine-tuned adapter: see "Hosting &
  deployment" in
  [`docs/spec-label-reader.md`](../docs/spec-label-reader.md) -- start
  with a single Hugging Face Space on ZeroGPU running both stages as a
  pipeline, confirm the combined latency fits the free daily quota before
  committing to it for daily use.
- FitTrack integration: new `api/analyze-label.js` endpoint and a "scan a
  label" UI flow in `food.html`/`js/food.js`, per the spec's integration
  section.